# Task A -- full-data R-Drop against a matched control

This is Run 5: does R-Drop improve the Task A recipe? Both arms are trained once on all
6,401 deduplicated labelled rows and then predict the official validation inputs. No
holdout and no OOF pass, so neither arm has a local macro-F1 and neither can have one.

| setting | value |
|---|---|
| data | all 6,401 deduplicated rows, `--folds 1` |
| encoder | `google/muril-base-cased`, stock |
| preprocessing | demojized, the Task A default |
| reinitialization | two top encoder layers, matching `run_rdrop.sh` |
| epochs | 6 |
| batch | 8 with 2 accumulation steps, effective 16 |
| seeds | 42, 43, 44, 45, 46; probabilities averaged |
| the one difference | `--rdrop 0` for the control, `--rdrop 0.5` for the variant |

## Why a control is trained here rather than reusing 0.8187

Run 9, the current best Task A submission at `0.8187`, blends a full-fit SVM with a MuRIL
component trained on a **single seed**. Comparing a five-seed R-Drop arm against it would
change two things at once, and the 806-row validation set cannot separate them. The
control arm in this notebook is identical to the variant in every respect except the
R-Drop flag, so whatever CodaBench reports between them is attributable to R-Drop alone.

## What it writes

Four ZIPs. The two MuRIL arms on their own, and each blended with the full-fit TF-IDF SVM
at Run 8's fixed 57/43 weights, so the result is also comparable with the current best.

| ZIP | contents |
|---|---|
| `task_a_control_full.zip` | five-seed MuRIL, no R-Drop |
| `task_a_rdrop_full.zip` | five-seed MuRIL, R-Drop 0.5 |
| `task_a_control_blend.zip` | the above control blended 43/57 with the SVM |
| `task_a_rdrop_blend.zip` | the R-Drop arm blended 43/57 with the SVM |

## Before you start

About **6.5 hours**: roughly 33 minutes per control seed and 45 per R-Drop seed, since
R-Drop adds a second forward pass. The SVM takes seconds.

Set **Accelerator** to `GPU T4 x2` or `GPU P100` and **Internet** on, then
**Save Version -> Save & Run All**. Never run this interactively: the session dies with
the browser tab.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
from hastika.common.preprocessing import dedupe_index

assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))
train = pd.read_csv("data/raw/binary_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Label"].tolist(), "task A")
print(f"raw labelled rows: {len(train)}; deduplicated rows used for fitting: {len(keep)}")
assert len(keep) == 6401, len(keep)

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Train the TF-IDF component once

`--full-fit` skips the OOF pass and fits on all 6,401 rows. It is shared by both blends,
so it runs once. `--demojize` matches the Task A convention established in Experiment 4.

In [ ]:
SVM_TAG = "task_a_svm_full_rdrop_expt"
run([sys.executable, "-u", "-m", "hastika.models.baseline_svm",
     "--task", "a", "--tag", SVM_TAG, "--demojize", "--full-fit"],
    log=f"artifacts/logs/{SVM_TAG}.log")
svm_probs = np.load(pathlib.Path("artifacts/runs") / SVM_TAG / "test_probs.npy")
print("svm test probabilities:", svm_probs.shape)

## 2. Train both MuRIL arms on all labelled data

`--folds 1` is a true full-data fit: every seed trains on all 6,401 rows and no row is
held back, which is why no score is printed. The two commands differ only in `--rdrop`.

Each arm averages five seeds. The next cell asserts that all five actually ran, because a
silently truncated arm would look like a legitimate result.

In [ ]:
SEEDS = ["42", "43", "44", "45", "46"]
COMMON = ["--model", "google/muril-base-cased", "--folds", "1", "--epochs", "6",
          "--bs", "8", "--grad-accum", "2", "--eval-bs", "32",
          "--select", "last", "--reinit-layers", "2", "--seeds", *SEEDS]
ARMS = [("task_a_control_full", ["--rdrop", "0"]),
        ("task_a_rdrop_full",   ["--rdrop", "0.5"])]

for tag, extra in ARMS:
    run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", tag,
         *COMMON, *extra], log=f"artifacts/logs/{tag}.log")

import re
for tag, _ in ARMS:
    text = pathlib.Path(f"artifacts/logs/{tag}.log").read_text()
    fits = re.findall(r"seed (\d+) FULL FIT, (\d+) rows", text)
    print(tag, fits)
    assert [s for s, _ in fits] == SEEDS, f"{tag}: not all five seeds ran"
    assert all(int(n) == 6401 for _, n in fits), f"{tag}: a seed did not see all rows"

## 3. Blend each arm with the SVM at Run 8's fixed weights

The weights are 57% SVM and 43% MuRIL, carried unchanged from the five-fold experiment in
Run 8. They are deliberately not re-optimized: the hidden validation labels must not
influence a final fit, and there is no OOF pass here to fit them on honestly.

In [ ]:
W_SVM, W_MURIL = 0.57, 0.43
ids = pd.read_csv("data/raw/binary_validation_inputs.csv")
blends = {}
for tag, _ in ARMS:
    muril_probs = np.load(pathlib.Path("artifacts/runs") / tag / "test_probs.npy")
    assert muril_probs.shape == svm_probs.shape, (muril_probs.shape, svm_probs.shape)
    blends[tag] = W_SVM * svm_probs + W_MURIL * muril_probs

for tag, _ in ARMS:
    out = pathlib.Path("artifacts/runs") / f"{tag}_blend"
    out.mkdir(parents=True, exist_ok=True)
    labels = np.where(blends[tag][:, 1] > 0.5, "Hate", "Non-Hate")
    pd.DataFrame({"id": ids["id"], "label": labels}).to_csv(out / "predictions.csv",
                                                            index=False)
    print(f"{tag}_blend", pd.Series(labels).value_counts().to_dict())

## 4. Package all four submissions

`hastika.common.submission` refuses to write unless the header is `id,label`, every id in
`binary_validation_inputs.csv` appears exactly once, and every label is `Hate` or
`Non-Hate`. Build every ZIP with it rather than by hand.

In [ ]:
ZIPS = []
for tag, _ in ARMS:
    for name, pred in [(tag, pathlib.Path("artifacts/runs") / tag / "predictions.csv"),
                       (f"{tag.replace('_full', '')}_blend",
                        pathlib.Path("artifacts/runs") / f"{tag}_blend" / "predictions.csv")]:
        z = f"/kaggle/working/{name}.zip"
        run([sys.executable, "-m", "hastika.common.submission",
             "--task", "a", "--pred", str(pred), "--out", z])
        with zipfile.ZipFile(z) as f:
            assert f.namelist() == ["predictions.csv"], f.namelist()
        ZIPS.append(pathlib.Path(z))
print("\nready to upload:")
for z in ZIPS:
    print(f"  {z.name:28s} {z.stat().st_size:6d} bytes")

## 5. Preserve downloadable outputs

Download these individually rather than using Download All. Keep the `test_probs.npy`
files: they are the only way to rebuild a blend later without retraining.

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_rdrop_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for z in ZIPS:
    shutil.copy2(z, OUT / z.name)
for tag, _ in ARMS:
    for name in ["predictions.csv", "test_probs.npy"]:
        p = pathlib.Path("artifacts/runs") / tag / name
        if p.exists():
            shutil.copy2(p, OUT / f"{tag}_{name}")
    shutil.copy2(f"artifacts/logs/{tag}.log", OUT / f"{tag}.log")
shutil.copy2(pathlib.Path("artifacts/runs") / SVM_TAG / "test_probs.npy",
             OUT / "svm_test_probs.npy")
print(sorted(p.name for p in OUT.iterdir()))

## 6. After CodaBench scores the submissions

Upload `task_a_control_blend.zip` first. It is the closest thing to a rerun of the current
best, so its score against `0.8187` tells you how much the 806-row validation set moves on
its own. Only then is the control-versus-R-Drop gap readable.

Record every score in `docs/EXPERIMENTS.md` and `submissions/README.md`, including the
arms that lose. Keep the current best as the candidate unless an R-Drop arm beats it by
more than the gap the control just revealed.

A note on what this notebook cannot tell you: with no holdout, there is no local number to
check before submitting. `experiments/task_a/run_rdrop.sh` is the five-fold version of the
same comparison and does produce an OOF score, at roughly five times the cost.